# MLB - Best Hitter by Ballpark
- Consists of a 100-game sample from the 2018 season

### Import libraries & env variables to use

In [1]:
# Import Necessary Libraries
import pandas as pd
import matplotlib
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [2]:
# Load env variables into Python
load_dotenv()

# Set up env variables to use
user = os.getenv("DB_USER")
if user is None: 
     raise ValueError("DB_USER not found in environment variables")
else: print("DB_USER loaded successfuly")

password = os.getenv("DB_PASSWORD")
if password is None:    
    raise ValueError("DB_PASSWORD not found in environment variables")
else: print("DB_PASSWORD loaded successfuly")

host = os.getenv("DB_HOST")
if host is None: 
    raise ValueError("DB_HOST not found in environment variables")
else: print("DB_HOST loaded successfuly")

port = os.getenv("DB_PORT")
if port is None: 
    raise ValueError("DB_PORT not found in environment variables")
else: print("DB_PORT loaded successfuly")

dbname = os.getenv("DB_NAME")
if dbname is None: 
    raise ValueError("DB_NAME not found in environment variables")
else: print("DB_NAME loaded successfuly")

DB_USER loaded successfuly
DB_PASSWORD loaded successfuly
DB_HOST loaded successfuly
DB_PORT loaded successfuly
DB_NAME loaded successfuly


### Get data from sql/base_query.sql into a DataFrame

In [3]:
# Create db connection 
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

In [4]:
# Read in SQL file
with open("../sql/base_query.sql") as f: 
    query = f.read()

In [5]:
# Create DataFrame using SQL file
df = pd.read_sql(query, engine)

In [6]:
# Get a sense of the DataFrame
df.head(10)

,g_id,ab_id,batter_id,position,first_name,last_name,team,event,out,strikeout,hit,homerun,home_team,away_team,venue_name
0,201800050,2018003856,594809,fielder,Adam,Eaton,WAS,Single,False,False,True,False,CIN,WAS,Great American Ball Park
1,201800050,2018003857,543685,fielder,Anthony,Rendon,WAS,Home Run,False,False,True,True,CIN,WAS,Great American Ball Park
2,201800050,2018003858,547180,fielder,Bryce,Harper,WAS,Groundout,True,False,False,False,CIN,WAS,Great American Ball Park
3,201800050,2018003859,475582,fielder,Ryan,Zimmerman,WAS,Groundout,True,False,False,False,CIN,WAS,Great American Ball Park
4,201800050,2018003860,607208,fielder,Trea,Turner,WAS,Walk,False,False,False,False,CIN,WAS,Great American Ball Park
5,201800050,2018003861,572191,fielder,Michael A.,Taylor,WAS,Strikeout,True,True,False,False,CIN,WAS,Great American Ball Park
6,201800050,2018003862,571740,fielder,Billy,Hamilton,CIN,Strikeout,True,True,False,False,CIN,WAS,Great American Ball Park
7,201800050,2018003863,553993,fielder,Eugenio,Suarez,CIN,Strikeout,True,True,False,False,CIN,WAS,Great American Ball Park
8,201800050,2018003864,458015,fielder,Joey,Votto,CIN,Single,False,False,True,False,CIN,WAS,Great American Ball Park
9,201800050,2018003865,594807,fielder,Adam,Duvall,CIN,Flyout,True,False,False,False,CIN,WAS,Great American Ball Park


### Cleanup & Feature Generation

In [7]:
# Combine name columns
df["player_name"] = (df["first_name"] + ' ' + df["last_name"])
df.head()

,g_id,ab_id,batter_id,position,first_name,last_name,team,event,out,strikeout,hit,homerun,home_team,away_team,venue_name,player_name
0,201800050,2018003856,594809,fielder,Adam,Eaton,WAS,Single,False,False,True,False,CIN,WAS,Great American Ball Park,Adam Eaton
1,201800050,2018003857,543685,fielder,Anthony,Rendon,WAS,Home Run,False,False,True,True,CIN,WAS,Great American Ball Park,Anthony Rendon
2,201800050,2018003858,547180,fielder,Bryce,Harper,WAS,Groundout,True,False,False,False,CIN,WAS,Great American Ball Park,Bryce Harper
3,201800050,2018003859,475582,fielder,Ryan,Zimmerman,WAS,Groundout,True,False,False,False,CIN,WAS,Great American Ball Park,Ryan Zimmerman
4,201800050,2018003860,607208,fielder,Trea,Turner,WAS,Walk,False,False,False,False,CIN,WAS,Great American Ball Park,Trea Turner


In [8]:
# Create new total_points feature to get a points column based on hits + (homerun *3) - strikeouts 
df["performance_points"] = df["hit"] + (df["homerun"] *3) - df["strikeout"]

### Data Analysis

In [9]:
# Create a new grouped DataFrame and aggragate the sum of certain columns
batter_game_df = df.groupby([
    "g_id",
    "batter_id",
    "player_name",
    "venue_name"
]).agg({
    "hit": "sum",
    "homerun": "sum",
    "strikeout": "sum",
    "ab_id": "count",
    "performance_points": "sum",
}).reset_index()

batter_game_df

,g_id,batter_id,player_name,venue_name,hit,homerun,strikeout,ab_id,performance_points
0,201800050,458015,Joey Votto,Great American Ball Park,1,0,0,4,1
1,201800050,461829,Gio Gonzalez,Great American Ball Park,0,0,2,3,-2
2,201800050,471083,Miguel Montero,Great American Ball Park,0,0,0,4,0
3,201800050,475582,Ryan Zimmerman,Great American Ball Park,1,0,1,5,0
4,201800050,519023,Devin Mesoraco,Great American Ball Park,1,0,1,4,0
...,...,...,...,...,...,...,...,...,...
2100,201802422,642086,Dominic Smith,Citi Field,0,0,1,3,-1
2101,201802422,642423,Magneuris Sierra,Citi Field,1,0,0,3,1
2102,201802422,642708,Amed Rosario,Citi Field,0,0,1,3,-1
2103,201802422,643446,Jeff McNeil,Citi Field,2,0,0,4,2


In [10]:
# Update column names to reflect aggs
batter_game_df = batter_game_df.rename(columns={
    "venue_name": "ballpark",
    "hit": "total_hits",
    "homerun": "total_homeruns",
    "strikeout": "total_strikeouts", 
    "total_atbats": "total_at_bats",
    "ab_id": "total_at_bats"
})

batter_game_df

,g_id,batter_id,player_name,ballpark,total_hits,total_homeruns,total_strikeouts,total_at_bats,performance_points
0,201800050,458015,Joey Votto,Great American Ball Park,1,0,0,4,1
1,201800050,461829,Gio Gonzalez,Great American Ball Park,0,0,2,3,-2
2,201800050,471083,Miguel Montero,Great American Ball Park,0,0,0,4,0
3,201800050,475582,Ryan Zimmerman,Great American Ball Park,1,0,1,5,0
4,201800050,519023,Devin Mesoraco,Great American Ball Park,1,0,1,4,0
...,...,...,...,...,...,...,...,...,...
2100,201802422,642086,Dominic Smith,Citi Field,0,0,1,3,-1
2101,201802422,642423,Magneuris Sierra,Citi Field,1,0,0,3,1
2102,201802422,642708,Amed Rosario,Citi Field,0,0,1,3,-1
2103,201802422,643446,Jeff McNeil,Citi Field,2,0,0,4,2


In [11]:
# Who are the top 10 batters agnostic of venue?
top_10_overall = batter_game_df.sort_values(by="performance_points", ascending=False).head(10)
top_10_overall


,g_id,batter_id,player_name,ballpark,total_hits,total_homeruns,total_strikeouts,total_at_bats,performance_points
29,201800073,544369,Didi Gregorius,Yankee Stadium,4,2,0,5,10
1421,201801790,660670,Ronald Acuna,SunTrust Park,3,2,0,5,9
1393,201801775,592206,Nicholas Castellanos,Comerica Park,5,1,0,5,8
697,201800940,500874,Jose Martinez,Great American Ball Park,2,2,0,4,8
6,201800050,547180,Bryce Harper,Great American Ball Park,2,2,0,5,8
1376,201801743,598265,Jackie Bradley,Oriole Park at Camden Yards,2,2,0,4,8
451,201800755,571697,Scooter Gennett,Coors Field,5,1,0,6,8
1277,201801628,624577,Yasiel Puig,Dodger Stadium,3,2,1,5,8
656,201800886,518692,Freddie Freeman,Petco Park,4,1,0,5,7
1701,201802026,542921,Tim Beckham,Kauffman Stadium,4,1,0,4,7


In [12]:
# Break down sorting & grouping logic into clear steps 
sorted_df = batter_game_df.sort_values(
    by="performance_points", 
    ascending=False
)

selected_cols = sorted_df[[
    "ballpark", 
    "player_name",
    "total_hits", 
    "total_homeruns", 
    "total_strikeouts", 
    "total_at_bats", 
    "performance_points"
]]

top_batter_by_ballpark = (selected_cols
    .groupby(by="ballpark")
    .head(1)
    .sort_values("ballpark")
)

top_batter_by_ballpark

,ballpark,player_name,total_hits,total_homeruns,total_strikeouts,total_at_bats,performance_points
1600,AT&T Park,Rougned Odor,3,1,0,4,6
1045,Angel Stadium,Kole Calhoun,3,1,0,5,6
375,Angel Stadium of Anaheim,Brian Dozier,4,1,0,4,7
1819,Busch Stadium,Manny Machado,3,1,0,5,6
522,Chase Field,Nick Ahmed,2,1,0,4,5
1063,Citi Field,Brandon Nimmo,1,1,0,1,4
65,Citizens Bank Park,Carlos Santana,2,1,0,4,5
1393,Comerica Park,Nicholas Castellanos,5,1,0,5,8
451,Coors Field,Scooter Gennett,5,1,0,6,8
1277,Dodger Stadium,Yasiel Puig,3,2,1,5,8


### Visually improve final table


In [13]:
# Create clean DataFrame to work from
final_output = top_batter_by_ballpark[[
    "ballpark",
    "player_name",
    "total_hits",
    "total_homeruns",
    "total_strikeouts",
    "total_at_bats",
    "performance_points"
]]

In [15]:
# Final clean DataFrame
(
    final_output.style
    .background_gradient(subset=["performance_points"], cmap="Greens")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("text-align", "center")]}
    ])
    .hide(axis="index")
)

ballpark,player_name,total_hits,total_homeruns,total_strikeouts,total_at_bats,performance_points
AT&T Park,Rougned Odor,3,1,0,4,6
Angel Stadium,Kole Calhoun,3,1,0,5,6
Angel Stadium of Anaheim,Brian Dozier,4,1,0,4,7
Busch Stadium,Manny Machado,3,1,0,5,6
Chase Field,Nick Ahmed,2,1,0,4,5
Citi Field,Brandon Nimmo,1,1,0,1,4
Citizens Bank Park,Carlos Santana,2,1,0,4,5
Comerica Park,Nicholas Castellanos,5,1,0,5,8
Coors Field,Scooter Gennett,5,1,0,6,8
Dodger Stadium,Yasiel Puig,3,2,1,5,8


## Final Findings

- Created a batter-game level dataset by aggregating at-bat data
- Developed a custom scoring metric:  
  **performance_points = hits + (3 × home runs) - strikeouts**

- Identified the best single-game hitting performance at each MLB ballpark

### Key Observations

- The top performances are heavily driven by home runs due to their higher weight in the scoring model
- Several players appear with strong multi-hit and power performances (e.g., 2 HR games)
- There is variation across ballparks, though this may be influenced by the limited 100-game sample

### Notes

- This analysis uses a subset of 2018 data, so results are illustrative rather than definitive
- A larger dataset would provide more reliable insights into ballpark-specific performance trends

### Top Overall Performance
- Didi Gregorius at Yankee Stadium recorded the highest performance_points (10)